# HDB Resale Flat Prices - Pipeline Orchestration

Runs the pipeline stages in sequence and sends a status alert at the end:

```
ingestion_to_source -> raw_iceberg -> data_profiling -> cleaned_iceberg -> transformed_iceberg -> hashed_iceberg -> send_alert
```

`data_profiling` (job_2b) profiles `raw_iceberg` (null rates, distinct values,
numeric quartiles, composite-key duplicate counts) and writes a JSON +
Markdown report to S3 - this is the Data Quality Requirement #2 deliverable,
and its output is the documented statistical basis for job_3's field
validation rules. It runs after `raw_iceberg` and before `cleaned_iceberg`.

All shared settings (Glue database, Athena workgroup, S3 buckets, the SNS
topic ARN, etc.) live in `config.py` - the pipeline's single source of
truth - and are imported here rather than duplicated. See `config.py`'s
docstring: hardcoding one of those values anywhere else is considered a bug.

## Two ways to run this notebook

Set `RUN_MODE` in the next cell to one of:

- **`"local"`** (default) - calls each job's `main()` function directly, in
  this notebook's own Python process, against whatever `common.py` /
  `config.py` are pointed at (real AWS resources via your local AWS
  credentials, or resources you've overridden via `HDB_*` environment
  variables - e.g. pointed at a scratch Glue database/S3 prefix for a dry
  run). No AWS Glue jobs need to exist yet. This is the mode a reviewer
  with only a checked-out repo (and appropriate AWS credentials/config) can
  run end-to-end without any deployment step.
- **`"glue"`** - triggers the 5 already-deployed AWS Glue jobs via `boto3`
  (`start_job_run`) and polls each to a terminal state before starting the
  next, exactly as this notebook did previously. Use this for the actual
  production/scheduled run once the jobs are deployed to Glue.

Both modes share the same step list, the same fail-fast behaviour (stop and
alert on the first failed step rather than continuing on incomplete
upstream data), and the same final SNS alert.

**Prerequisites**

- **For `RUN_MODE = "local"`**: this notebook sits in the same folder as
  `config.py`, `common.py`, `job_1..job_5`, and `job_2b_data_profiling.py`
  (the `hdb/` project folder), so it can `import` them directly. Your AWS
  credentials (`~/.aws/credentials` or environment variables) need access
  to whatever S3 buckets / Glue Data Catalog / Athena workgroup / SNS topic
  `config.py` resolves to - override any of those via `HDB_*` environment
  variables (see `config.py`'s docstring) to point at scratch/dev resources
  before running.
- **For `RUN_MODE = "glue"`**: the 6 Glue jobs (`job_1` .. `job_5`, plus the
  profiling job) are already created in AWS Glue, pointing at the scripts
  wherever they're deployed (see `PIPELINE_SCRIPTS_S3_BUCKET` in
  `config.py`), with `common.py` and `config.py` uploaded as Python library
  references (`--extra-py-files`) or packaged alongside.
- An SNS topic exists for alerts; its ARN is `config.SNS_TOPIC_ARN`
  (override via the `HDB_SNS_TOPIC_ARN` environment variable, same as every
  job resolves it - do not hardcode a different ARN here).


## Deploy (optional)

Runs `setup.sh` before triggering the pipeline steps below. `setup.sh` is idempotent - it
checks for existing S3 buckets, the Glue database, IAM roles, and the SNS topic before
creating anything, and re-uploads the current `pipeline-scripts/` tree (including this
`ETL/` folder) to S3 either way. So every run of this notebook deploys the latest code
first, whether `RUN_MODE` below is `"local"` or `"glue"` - for `"glue"` this matters a lot
(Glue reads scripts from S3, so a stale S3 copy means Glue runs stale code even after you've
edited `job_*.py` locally); for `"local"` the deploy step is mostly just keeping S3 in sync
for whenever you do switch to `"glue"`.

Set `RUN_SETUP = False` in the next cell to skip this and go straight to the pipeline run
(e.g. you already ran `setup.sh` moments ago and don't want to re-check/re-upload everything).

**Assumes:** `setup.sh` sits two directories up from this notebook (`hdb_1/setup.sh`, with
this notebook at `hdb_1/pipeline-scripts/ETL/`), and that `bash` is on your system PATH
(true for Git Bash, which is what you've been running AWS CLI commands from).


In [1]:
import subprocess
from pathlib import Path

RUN_SETUP = True  # False to skip deployment and go straight to the pipeline run below
SETUP_SCRIPT = Path("../../setup.sh").resolve()

if RUN_SETUP:
    if not SETUP_SCRIPT.exists():
        raise FileNotFoundError(
            f"setup.sh not found at {SETUP_SCRIPT} - adjust the path above if you've moved "
            "this notebook or setup.sh."
        )

    print(f"Deploying: running {SETUP_SCRIPT}\n")
    # check=True -> raises subprocess.CalledProcessError on any non-zero exit, which stops
    # this notebook here rather than proceeding to run the pipeline against a partial/failed
    # deploy. Output streams straight to this cell since we don't capture it.
    subprocess.run(["bash", str(SETUP_SCRIPT)], cwd=SETUP_SCRIPT.parent, check=True)
    print("\nDeploy completed: buckets/roles/database/topic verified, latest scripts synced to S3.")
else:
    print("Skipping setup.sh (RUN_SETUP=False) - assuming infrastructure and scripts are already up to date.")


Deploying: running C:\Users\ssuje\OneDrive\Desktop\claude\hdb_1\setup.sh



CalledProcessError: Command '['bash', 'C:\\Users\\ssuje\\OneDrive\\Desktop\\claude\\hdb_1\\setup.sh']' returned non-zero exit status 1.

In [ ]:
import sys
import time

import boto3

# This notebook assumes it sits alongside config.py / common.py / job_*.py
# (the hdb/ project folder). Adjust this path if you keep it somewhere else.
sys.path.insert(0, ".")

from config import AWS_REGION, SNS_TOPIC_ARN

# --------------------------------------------------------------------------- #
# Run mode
# --------------------------------------------------------------------------- #
# "local" - call each job's main() in-process, no Glue deployment required.
# "glue"  - trigger already-deployed AWS Glue jobs via boto3 and poll them.
RUN_MODE = "local"  # "local" | "glue"

assert RUN_MODE in ("local", "glue"), f"RUN_MODE must be 'local' or 'glue', got {RUN_MODE!r}"

sns = boto3.client("sns", region_name=AWS_REGION)
glue = boto3.client("glue", region_name=AWS_REGION) if RUN_MODE == "glue" else None

# Whether to run hashed_iceberg (job_5) as the final step of this chain.
# job_5 is fully implemented (hashing + SCD2 versioning), so it's on by
# default - set to False to trigger it independently instead.
INCLUDE_HASHED_STEP = True

# Ordered pipeline steps. data_profiling (job_2b) sits between raw_iceberg
# and cleaned_iceberg: it profiles raw_iceberg and its output is the
# statistical basis for cleaned_iceberg's field-validation rules.
PIPELINE_STEPS = [
    "ingestion_to_source",
    "raw_iceberg",
    "data_profiling",
    "cleaned_iceberg",
    "transformed_iceberg",
]
if INCLUDE_HASHED_STEP:
    PIPELINE_STEPS.append("hashed_iceberg")

# Map of pipeline step name -> actual Glue job name (as created in AWS Glue).
# Only used when RUN_MODE == "glue".
GLUE_JOB_NAMES = {
    "ingestion_to_source": "hdb-job-1-ingestion-to-source",
    "raw_iceberg":         "hdb-job-2-raw-iceberg",
    "data_profiling":      "hdb-job-2b-data-profiling",
    "cleaned_iceberg":     "hdb-job-3-cleaned-iceberg",
    "transformed_iceberg": "hdb-job-4-transformed-iceberg",
    "hashed_iceberg":      "hdb-job-5-hashed-iceberg",
}

# NOTE: SNS_TOPIC_ARN is imported from config.py above, not hardcoded here -
# config.py is the single source of truth for it (env override:
# HDB_SNS_TOPIC_ARN), matching how every job resolves it.

GLUE_POLL_INTERVAL_SECONDS = 15  # how often we poll Glue for job-run status
# (RUN_MODE == "glue" only) - distinct from config.POLL_INTERVAL_SECONDS,
# which is job_1's data.gov.sg API poll interval and unrelated to this loop.
TERMINAL_STATES = {"SUCCEEDED", "FAILED", "STOPPED", "TIMEOUT", "ERROR"}


In [ ]:
def run_glue_job(job_name: str) -> dict:
    """Start a Glue job run and block until it reaches a terminal state."""
    print(f"Starting Glue job: {job_name}")
    start_resp = glue.start_job_run(JobName=job_name)
    run_id = start_resp["JobRunId"]

    while True:
        status = glue.get_job_run(JobName=job_name, RunId=run_id)["JobRun"]
        state = status["JobRunState"]
        print(f"  [{job_name}] run_id={run_id} state={state}")
        if state in TERMINAL_STATES:
            return status
        time.sleep(GLUE_POLL_INTERVAL_SECONDS)


In [ ]:
# Local entry points, imported lazily inside run_local_step() rather than at
# the top of the notebook: importing all 6 job modules up front would import
# things like `requests`/`boto3` clients from job_1 even for a run that only
# needs, say, job_5 - and it keeps this cell's mapping self-contained.

def run_local_step(step_name: str) -> dict:
    """Call the matching job's main() directly in-process. Returns a dict
    shaped like the bits of a Glue job-run status this notebook actually
    reads (JobRunState / ErrorMessage), so run_step_or_alert() can treat
    both modes identically."""
    print(f"Running locally: {step_name}")
    try:
        if step_name == "ingestion_to_source":
            import job_1_ingestion_to_source as job
        elif step_name == "raw_iceberg":
            import job_2_raw_iceberg as job
        elif step_name == "data_profiling":
            import job_2b_data_profiling as job
        elif step_name == "cleaned_iceberg":
            import job_3_cleaned_iceberg as job
        elif step_name == "transformed_iceberg":
            import job_4_transformed_iceberg as job
        elif step_name == "hashed_iceberg":
            import job_5_hashed_iceberg as job
        else:
            raise ValueError(f"Unknown step: {step_name}")

        job.main()
        print(f"  [{step_name}] SUCCEEDED")
        return {"JobRunState": "SUCCEEDED"}

    except Exception as exc:
        print(f"  [{step_name}] FAILED: {exc}")
        return {"JobRunState": "FAILED", "ErrorMessage": str(exc)}


In [ ]:
def send_alert(subject: str, message: str) -> None:
    sns.publish(TopicArn=SNS_TOPIC_ARN, Subject=subject[:100], Message=message)
    print(f"Alert sent: {subject}")


## Run the chain

Each step only proceeds if the previous one `SUCCEEDED`. Any non-success
state stops the notebook and sends a failure alert immediately, rather than
continuing to run downstream stages on incomplete upstream data.


In [ ]:
run_log = []

def run_step_or_alert(step_name: str) -> bool:
    if RUN_MODE == "glue":
        job_name = GLUE_JOB_NAMES[step_name]
        result = run_glue_job(job_name)
    else:
        job_name = f"local:{step_name}"
        result = run_local_step(step_name)

    state = result["JobRunState"]
    run_log.append((step_name, job_name, state))

    if state != "SUCCEEDED":
        error_msg = result.get("ErrorMessage", "No error message returned.")
        send_alert(
            subject=f"HDB pipeline FAILED at step: {step_name}",
            message=(
                f"Step '{step_name}' ('{job_name}', mode={RUN_MODE}) ended in state {state}.\n"
                f"Error: {error_msg}\n\n"
                f"Run log so far: {run_log}"
            ),
        )
        return False
    return True


In [ ]:
pipeline_succeeded = True
for step in PIPELINE_STEPS:
    if not run_step_or_alert(step):
        pipeline_succeeded = False
        break

if pipeline_succeeded:
    send_alert(
        subject="HDB pipeline completed successfully",
        message=f"Mode: {RUN_MODE}\nAll steps completed:\n{run_log}",
    )

print("\nFinal run log:")
for step_name, job_name, state in run_log:
    print(f"  {step_name:22s} ({job_name:30s}) -> {state}")
